# VERITAS 08 — Evaluation

**Phase 15.** The table that decides whether the architecture earns its
complexity.

## The ablation ladder

| system | what it adds |
|---|---|
| **B1 LLM-only** | nothing — measures what the weights memorised |
| **B2 Basic RAG** | dense top-k |
| **B3 Hybrid RAG** | + BM25 + fusion + reranking |
| **B4 Temporal RAG** | + freshness and validity signals (the strongest *published* pattern — the honest bar) |
| **VERITAS** | + bitemporal store, agentic loop, claim verification, contradiction analysis, abstention |

Every row uses the **same corpus, tokenizer and weights**. Only the pipeline
differs. Comparing against a differently-trained model would confound "my
architecture is better" with "my model is bigger".

**B4 → VERITAS is the row that carries the claim.** If VERITAS does not beat
temporal RAG on CONFLICT, INSUFFICIENT and OUTDATED_SOURCE, the extra machinery
is not earning its keep — and the honest thing is to report that.

## Why a purpose-built benchmark

NQ/HotpotQA/TriviaQA assume a static corpus with one right answer. They have no
label for *"Person A was correct in 2024, wrong now"*, none for *"the sources
disagree, say so"*, none for *"abstain"* — and a system quoting a stale source
scores **identically** to one quoting the current source, as long as the string
matches. Those missing labels are the entire subject of this project.

In [ ]:
import sys, os, time, math, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, torch
torch.manual_seed(1337); np.random.seed(1337); random.seed(1337)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from veritas.tokenizer.bpe import BPETokenizer
from veritas.model.transformer import VeritasLM, ModelConfig
from veritas.pipeline import VeritasSystemBuilder, load_benchmark_corpus
from veritas.eval.benchmark import build_seed_benchmark, expand_synthetic, Category
from veritas.eval.baselines import (LLMOnly, BasicRAG, HybridRAG, TemporalRAG,
                                    VeritasSystem, evaluate, compare, freshness_lag)
from veritas.eval.metrics import format_table

bench = expand_synthetic(build_seed_benchmark(), 5)
print('benchmark:', bench.stats())
bench.save(ROOT/'data'/'bench'/'temporal_evidence_bench.json')

In [ ]:
tok = BPETokenizer.load(ROOT/'checkpoints'/'tokenizer.json')
ck = ROOT/'checkpoints'/'sft.pt'
if not ck.exists(): ck = ROOT/'checkpoints'/'best.pt'
model = VeritasLM.load(ck, DEVICE) if ck.exists() else VeritasLM(
    ModelConfig(vocab_size=tok.vocab_size, d_model=384, n_layers=8, n_heads=8,
                n_kv_heads=2, max_seq_len=256)).to(DEVICE)
model.eval()
if not ck.exists():
    print('WARNING: no trained checkpoint. The embedder is random, so dense retrieval')
    print('is noise and the table below is not reportable. Run notebooks 03-05 first.')

b = VeritasSystemBuilder(model, tok, device=DEVICE)
load_benchmark_corpus(b, bench)
from veritas.agents.orchestrator import VeritasConfig
veritas = b.build(config=VeritasConfig(verbose=False, domain='corporate'))
print('corpus:', len(b.corpus), 'chunks |', b.ingest.summary()['versions'], 'fact versions')

In [ ]:
systems = [LLMOnly(model, tok, DEVICE),
           BasicRAG(b.vec, b.embedder, b.corpus, b.metadata),
           HybridRAG(b.retriever(), b.corpus, b.metadata),
           TemporalRAG(b.retriever(), b.corpus, b.metadata),
           VeritasSystem(veritas)]
print(compare(systems, bench))

## Reading the table honestly

* **accuracy** is scored per category: abstention items need an abstention,
  conflict items need the conflict surfaced, CHANGE items need *every* state.
* **coverage / cite_acc** are structurally 0 for B1–B4. They do not verify
  claims, so there is nothing to measure — that is the point of the column, not
  a scoring trick.
* **conflict** counts true positives **and** true negatives. A system that
  never surfaces a conflict scores well on the 7 non-conflict items, so read
  this column together with the CONFLICT row of the category table.
* **abstain_f1** balances correct abstention against over-abstention. Trading
  one for the other is the entire design question; a single number would hide
  it, which is why `AbstentionScore` also exposes both rates separately.

In [ ]:
m = evaluate(systems[-1], bench)
a = m.abstention
print(f'correct abstentions : {a.correct_abstentions}')
print(f'missed (answered when it should not have) : {a.missed_abstentions}')
print(f'over-abstentions (refused with evidence)  : {a.over_abstentions}')
print(f'precision {a.abstention_precision:.2f} | recall {a.abstention_recall:.2f} | F1 {a.balanced:.2f}')
print('\nBoth failure directions are reported. A system that always refuses gets')
print('recall 1.0 and is useless; precision is what catches that.')

## Per-category breakdown

This is the interesting plot. Look for the categories where the ladder is flat
until VERITAS — those are the capabilities the architecture actually adds.

In [ ]:
import matplotlib.pyplot as plt
results = [evaluate(s, bench) for s in systems]
cats = list(Category.ALL)
x = np.arange(len(cats)); w = 0.16
plt.figure(figsize=(13,4))
for i, r in enumerate(results):
    plt.bar(x + i*w, [r.by_category.get(c, 0) for c in cats], w, label=r.name)
plt.xticks(x + 2*w, cats, rotation=20, ha='right'); plt.ylabel('accuracy')
plt.title('Accuracy by question category'); plt.legend(fontsize=8); plt.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## Freshness lag — the metric a batch-rebuild system cannot produce

Answer, inject a new source, answer again. Report the wall-clock cost of
incorporating new information and whether the answer actually moved.

In [ ]:
r = freshness_lag(b.ingest, veritas,
    'Who is the current CEO of Acme Industries?', 'sec.gov',
    'Acme Industries filing: Yuki Tanaka was appointed chief executive effective March 2027.',
    'Acme Industries', '2027-03-02')
for k_, v_ in r.items():
    print(f'{k_:22s}: {str(v_)[:120]}')
print('\nA nightly-rebuild pipeline cannot report a number here at all.')

## Retrieval metrics in isolation

Retrieval is the ceiling on everything downstream: evidence not retrieved
cannot be verified. **nDCG** is the headline number — it is the only one of the
four that handles graded relevance *and* position together.

In [ ]:
from veritas.eval.metrics import recall_at_k, precision_at_k, mrr, ndcg_at_k
rows = []
for s in systems[1:]:
    R = P = M = N = 0; n = 0
    for it in bench.items:
        if not it.gold_docs: continue
        _, retrieved = s.answer(it)
        docs, seen = [], set()
        for c in retrieved:
            d = c.split('#')[0]
            if d not in seen: seen.add(d); docs.append(d)
        R += recall_at_k(docs, it.gold_docs, 5); P += precision_at_k(docs, it.gold_docs, 5)
        M += mrr(docs, it.gold_docs); N += ndcg_at_k(docs, it.relevance_map, 10); n += 1
    rows.append({'system': s.name, 'recall@5': round(R/n,3), 'precision@5': round(P/n,3),
                 'MRR': round(M/n,3), 'nDCG@10': round(N/n,3)})
print(format_table(rows))

## What to claim, and what not to

Defensible from this table:

* VERITAS is the only configuration that **abstains** when evidence is absent,
  **surfaces** conflicts instead of silently picking, and **qualifies** stale
  evidence as stale.
* Claim-level coverage and citation accuracy are measurable for VERITAS and
  structurally undefined for the baselines.
* New information reaches answers in **milliseconds, without retraining**.

**Not** defensible:

* "This has never been done." Temporal RAG, bitemporal databases, FEVER-style
  claim verification and RAG self-verification all exist independently. See
  `docs/novelty.md` — the contribution is the *combination* plus the benchmark
  that measures it, and it should be stated that way.
* Any number from a run without a trained checkpoint. With a random encoder,
  dense retrieval is noise and the table swings run to run.
* Any absolute accuracy figure from an 8-item seed set. Expand the benchmark
  (`expand_synthetic`, or hand-write more items) before reporting.

In [ ]:
print('Reproduce everything:')
print('  python tests/test_smoke.py          # 9 correctness tests')
print('  python scripts/run_eval.py --synthetic 5 --checkpoint checkpoints/sft.pt')
print('\nNotebooks 01-08 build the whole system from an empty directory.')